# War and conflict

In [1]:
import polars as pl

# cfg = pl.Config()
# cfg.set_tbl_rows(30)
# cfg.set_fmt_str_lengths(1000)

## Get data

In [2]:
df = pl.read_parquet('../data/facts/FIN*.parquet')

In [3]:
df.schema

Schema([('entity_word_id', String),
        ('speech_id', String),
        ('paragraph_id', String),
        ('sentence_id', String),
        ('sentence_content_current', String),
        ('sentence_content_previous', String),
        ('sentence_content_next', String),
        ('sentence_sentiment_value', Float32),
        ('sentence_sentiment_ana', String),
        ('total_tokens_in_speech', Int32),
        ('total_tokens_in_session', Int32),
        ('entity_category', String),
        ('entity_content', String),
        ('country', String),
        ('session_date', Date),
        ('debate_topic', String),
        ('speaker_id', String),
        ('speaker_ana', String)])

In [4]:
# Number of rows
df.height

464872

In [5]:
# Annotate session year
df = df.with_columns(pl.col('session_date').dt.year().alias('session_year'))

# Number of rows per year
df.group_by('session_year').agg(pl.len().alias('nb_rows_per_year')).sort('session_year')

session_year,nb_rows_per_year
i32,u32
2015,42475
2016,77542
2017,74275
2018,65414
2019,60363
2020,77986
2021,66182
2022,635


In [6]:
# Number of rows per debate topic
df.group_by('debate_topic').agg(pl.len().alias('nb_rows_per_topic')).sort(
    'debate_topic'
)

debate_topic,nb_rows_per_topic
str,u32
"""argic""",15570
"""civil""",18286
"""cultu""",2095
"""defen""",18993
"""domes""",10866
…,…
"""other""",32095
"""techn""",6730
"""trade""",3897


## Filter data

In [7]:
def get_results(filtered_df):
    return (
        filtered_df.with_columns(
            pl.struct(
                [
                    pl.col('session_year'),
                    pl.col('debate_topic'),
                    pl.col('sentence_sentiment_value'),
                    pl.col('sentence_sentiment_ana'),
                    (
                        pl.col('sentence_content_previous').fill_null('')
                        + ' '
                        + pl.col('sentence_content_current')
                        + ' '
                        + pl.col('sentence_content_next').fill_null('')
                    ).str.strip_chars(),
                ]
            ).alias('sentence_with_context')
        )
        .sort('session_year')
        .group_by([pl.col('entity_category'), pl.col('entity_content')])
        .agg(pl.col('sentence_with_context').alias('sentences_with_context'))
        .with_columns(pl.col('sentences_with_context').list.len().alias('nb_sentences'))
        .sort('nb_sentences', descending=True)
    )


### Keywords

In [8]:
# Look for a particular keyword in "sentence_content_current"
def search_keyword(keyword):
    pattern = rf'(?i){keyword}'
    return get_results(
        df.filter(
            pl.col('entity_category').is_in(['LOC']),
            pl.col('sentence_content_current').str.contains(pattern),
        )
    )

In [9]:
# Mentions of "war"
search_keyword(r'\bwar\b')

entity_category,entity_content,sentences_with_context,nb_sentences
str,str,list[struct[5]],u32
"""LOC""","""Finland""","[{2015,""inter"",1.367,""senti:mixneg"",""When we look at the recent history of Finland , of course , this situation is dramatically new and partly the result of our being a Member State of the EU . For the first time since the war , Finland joined the anti-Russia political front clearly . Well , Finland has had its own interests under control , of course , in its relationship with Russia , it has been and will be , and not least on the commercial side , and these , of course , puzzled us how our commercial cooperation is going to be , how this will affect Finland when we are involved in these sanctions .""}, {2015,""inter"",4.994,""senti:pospos"",""Every single shot , without a front . And they paid the war compensation , and Finland left that postwar war and compensation and became a welfare state . We are now in a somewhat similar situation .""}, … {2021,""defen"",1.834,""senti:neuneg"",""The early warning period for crises has been reduced and the threshold for the use of force has decreased . Finland is not threatened by an immediate military crisis , but the wise prepares for everything — including situations where the barriers to war and peace are blurred . The Finnish security environment has remained tense and unpredictable .""}]",249
"""LOC""","""Europe""","[{2015,""inter"",1.544,""senti:neuneg"",""I am not going through the issues of election observation and many others with which the OSCE is trying to push the situation in Ukraine forward , but I say that 40 years from Helsinki to this day means that the OSCE , in a way , needs a new generation . We must achieve OSCE 2.0 , which will help Europe not to divide itself strongly into two hostile and opposite camps , in line with the legacy of the Second World War . Of course , the OSCE does not live its own life , but it is a reflection of international political reality , but in this reality there is now a deep concern about the division of Europe .""}, {2015,""immig"",4.009,""senti:mixpos"",""The Ministry of the Interior has responsibility for these so - called return agreements , and in this context it would be useful to highlight the fact that the Tuesday Council of Ministers was very far advanced in the debate on the establishment of a common EU list of safe countries , and this is being speeded up . In other words , if the EU still jointly defines those countries that are so - called safe and can be returned , such as the Balkan countries , then we can concentrate our resources on helping those people who are in real crisis areas and , more quickly again , as the Albanians have been returned this year , 600 people , more quickly to their own country , because together in Europe and in Finland we see that there is no such danger as , for example , in the area of the Syrian war .""}, … {2021,""inter"",0.019,""senti:negneg"",""The situation in Europe is currently extremely dangerous and critical . Russia is threatening , in practice , to start a war in Europe , a war against Ukraine . Of course , talking about the start of the war is somewhat misleading , as Russia attacked Ukraine as early as 2014 , when it illegally occupied the Crimean Peninsula and joined it and began an ongoing war in Eastern Ukraine .""}]",62
"""LOC""","""Russia""","[{2015,""inter"",0.131,""senti:negneg"",""However , it must be clearly stated that Russia 's action has caused the crisis in Ukraine . Russia has violated its agreements , including the Etyk Treaty , which was signed by its predecessor Soviet Union , Russia has attacked its neighbouring Ukraine , took over the Crimean Peninsula and is currently at war in Eastern Ukraine . At the same time , Russian propaganda lies to its own and to the rest of the world and even stirs up a kind of war psychology in Russia , where in propaganda NATO and the West are any day attacking Russia under the leadership of various fascists 

In [10]:
# Mentions of "conflict"
search_keyword('conflict')

entity_category,entity_content,sentences_with_context,nb_sentences
str,str,list[struct[5]],u32
"""LOC""","""Finland""","[{2015,""civil"",1.581,""senti:neuneg"",""The UN General Assembly has also adopted two additional protocols in 2000 . The second , which has entered into force in Finland in 2002 , concerns the participation of children in armed conflicts , and the Protocol on the sale of children , child prostitution and child pornography has entered into force in Finland in 2012 . Under this Agreement , the Parties , i.e. the States , are committed to taking steps to implement the Protocols and the rights guaranteed by this Agreement .""}, {2015,""civil"",1.581,""senti:neuneg"",""The UN General Assembly has also adopted two additional protocols in 2000 . The second , which has entered into force in Finland in 2002 , concerns the participation of children in armed conflicts , and the Protocol on the sale of children , child prostitution and child pornography has entered into force in Finland in 2012 . Under this Agreement , the Parties , i.e. the States , are committed to taking steps to implement the Protocols and the rights guaranteed by this Agreement .""}, … {2021,""inter"",3.505,""senti:mixpos"",""Mr President ! In addition to development cooperation , next year 's draft budget focuses on strengthening Finland 's network of representatives , supporting the availability of labour and participation in international crisis management , conflict prevention and mediation . The Ministry of Foreign Affairs will also continue to develop the Team Finland Export Promotion Network to support the business opportunities and employment of Finnish companies and prepare new solutions to promote exports and trade .""}]",221
"""LOC""","""Europe""","[{2015,""inter"",1.426,""senti:mixneg"",""Thank you to the chairman of the Finnish delegation , Mr Guzenina , for this report , which quite rightly reflects Finland 's active approach to this parliamentary cooperation . On the other hand , this report also shows , in a rather sad way , that there is also a need for this cooperation in these unfortunate conflicts and there is a great need for it in Europe , in the vicinity of Europe 's external borders and beyond . Many crises continue to exist , and new ones are emerging .""}, {2015,""inter"",1.426,""senti:mixneg"",""Thank you to the chairman of the Finnish delegation , Mr Guzenina , for this report , which quite rightly reflects Finland 's active approach to this parliamentary cooperation . On the other hand , this report also shows , in a rather sad way , that there is also a need for this cooperation in these unfortunate conflicts and there is a great need for it in Europe , in the vicinity of Europe 's external borders and beyond . Many crises continue to exist , and new ones are emerging .""}, … {2021,""defen"",1.107,""senti:mixneg"",""It is important that the adequacy of staff resources is ensured in all its different forms . Finland is situated in an area of strategic importance from the point of view of the major powers , and , unfortunately , in a possible Europe - wide conflict , the northern Europe would form an integrated whole in a military strategy . If security in our neighbourhood or elsewhere in Europe were threatened , Finland as a Member State of the European Union could not exclude itself .""}]",32
"""LOC""","""Iraq""","[{2015,""immig"",3.965,""senti:mixpos"",""Turkey is at the heart of this major refugee and migration crisis , and cooperation is also needed here . I spoke with the Russian Foreign Minister , Sergei Lavrov , and we said together that wide - ranging international cooperation aimed specifically at resolving the conflict in Syria , the solution of the conflict in Iraq , is the way in which this is most effective . Then as far as safe countries are concerned — just as you said , Bosnia and Herzegovina , Albania , other such :""}, {2016,""inter"",2.694,""senti:neupos"",""Work with partner countries is long - term

### Entities

In [11]:
# Look for a particular entity in "entity_content"
def search_entity(entity):
    pattern = rf'(?i){entity}'
    return get_results(df.filter(pl.col('entity_content').str.contains(pattern)))

In [12]:
# Mentions of "ukrain"
search_entity('ukrain')

entity_category,entity_content,sentences_with_context,nb_sentences
str,str,list[struct[5]],u32
"""LOC""","""Ukraine""","[{2015,""civil"",2.727,""senti:neupos"",""Today , the Council of Europe consists of 47 countries . These include Russia and Ukraine . More than 700 MPs from different countries participate in the Parliamentary Assembly , including members and alternates .""}, {2015,""civil"",1.664,""senti:neuneg"",""This report by the Council of Europe for 2014 highlights one thing above all . Last year 's biggest issue at the General Assembly of the Council of Europe in the member committees was the situation in Ukraine , Russia 's actions and the related challenges . This issue is still very topical .""}, … {2021,""inter"",3.91,""senti:mixpos"",""Of course we encourage the parties , especially Russia , to fulfil the obligations of the Minsk Agreement . Of course , in this situation , we are trying to prevent the emergence of a new division and we want to highlight the principle that we respect Ukraine 's independence and its territorial integrity , as is the territorial integrity of any other European country in these situations . It is also good to bring out those principles at this very beginning when this communication is taking place .""}]",441
"""LOC""","""Eastern Ukraine""","[{2015,""inter"",0.169,""senti:negneg"",""Whatever mistakes Ukraine has made in strengthening its own society , it is quite clear that Russia 's action in Ukraine is unacceptable , Robbery of Crimea and war in Eastern Ukraine . It is great that the Nordic Council has condemned Russia 's actions .""}, {2015,""inter"",0.131,""senti:negneg"",""However , it must be clearly stated that Russia 's action has caused the crisis in Ukraine . Russia has violated its agreements , including the Etyk Treaty , which was signed by its predecessor Soviet Union , Russia has attacked its neighbouring Ukraine , took over the Crimean Peninsula and is currently at war in Eastern Ukraine . At the same time , Russian propaganda lies to its own and to the rest of the world and even stirs up a kind of war psychology in Russia , where in propaganda NATO and the West are any day attacking Russia under the leadership of various fascists and Nazis .""}, … {2021,""inter"",0.093,""senti:negneg"",""Russia is threatening , in practice , to start a war in Europe , a war against Ukraine . Of course , talking about the start of the war is somewhat misleading , as Russia attacked Ukraine as early as 2014 , when it illegally occupied the Crimean Peninsula and joined it and began an ongoing war in Eastern Ukraine . There is grotesque propaganda that Russia is currently presenting to its own people and to the whole world .""}]",64
"""MISC""","""Ukrainian""","[{2015,""civil"",2.79,""senti:neupos"",""There have also been issues such as the impact of the Internet on political activity , international assistance to Syrian refugees and the democratic development of Palestine . Reports have also been made on the functioning of Ukrainian democratic institutions and the new Spanish abortion law . That 's a lot of different things .""}, {2015,""inter"",3.369,""senti:neupos"",""For my part , the OSCE Parliamentary Assembly includes all 57 OSCE Member States , without any of these Member States being pointed at the door . And , of course , it is also necessary to keep this concept going because the OSCE , of course , plays a key role , more central than any other parliamentary body in terms of dialogue around the Ukrainian crisis — a dialogue that no other measure can replace in seeking peace , reconciliation and reconciliation and improving the conditions . In fact , when I listened to the text of the chairman of the Finnish delegation to the Council of Europe , I wondered how closely the General Assembly of the Council of Europe focuses on certain issues , how much overlap there is with the OSCE and how much diversity there is .""}, … {2021,""inter"",0.777,""senti:mixneg"",""Then there 's the res

In [13]:
# Mentions of "afghan"
search_entity('afghan')

entity_category,entity_content,sentences_with_context,nb_sentences
str,str,list[struct[5]],u32
"""LOC""","""Afghanistan""","[{2015,""inter"",0.528,""senti:mixneg"",""We want to help these people . What we do not want to do again is , for example , to give Afghanistan bilateral money — I think some EUR 8 million is being given this year , and last year money was given automatically . Bilateral development aid money we give around 300 million per year , as I recall , and a very large part of that money is going down the well in Kankkula .""}, {2015,""inter"",0.096,""senti:negneg"",""Bilateral development aid money we give around 300 million per year , as I recall , and a very large part of that money is going down the well in Kankkula . In Afghanistan , banks were caught when $ 900 million was stolen there in 2010 , and only a fraction of it was recovered , even though they promised to do everything they could . And yet , Finland continues to support such countries with direct cash .""}, … {2021,""inter"",4.582,""senti:pospos"",""All in all , high value added products must be at the centre . Representative Niikko , Afghanistan has achieved results . Congressman Tynkkynen , humanitarian aid alone is not enough . It 's a Band - Aid .""}]",869
"""MISC""","""Afghan""","[{2015,""inter"",1.541,""senti:neuneg"",""I 'd love to , and I do it on a regular basis . But it is quite different to talk about you putting a coin into the Afghan administration 's account or into the Palestinian administration 's account , and it is the same kind of transparent money as this church collection . They 're two different things . The difference is like night and day .""}, {2015,""defen"",2.319,""senti:neuneg"",""This government will continue to play an active role in international crisis management , but the framework conditions of the economy are severe . If the Afghan operation , Resolute Support , is decided to continue after the end of 2016 , it will most likely be carried out through an additional budget . I can assure you , also in accordance with the Government Programme , that this government will guarantee universal military service .""}, … {2021,""inter"",-0.052,""senti:negneg"",""Afghanistan became an expensive lesson for everyone . Tax money burned in sacks does not at present illustrate the future prospects of an Afghan girl . Even if there is no record of Christmas in the country , many girls are offered a soft package , which now reveals burka instead of woolen socks .""}]",109
"""MISC""","""Afghans""","[{2015,""inter"",0.625,""senti:mixneg"",""Syria and Iraq are key countries of origin for the refugee crisis affecting Europe . The continuing civil war in Afghanistan has increased the number of Afghans seeking to enter Europe , and the number of asylum seekers seeking to enter Europe from Africa has also remained high . A lasting solution to the refugee crisis will only be achieved by calming the situation in the crisis areas .""}, {2015,""inter"",1.375,""senti:mixneg"",""Mr President ! Mr Soini , I think you are right to judge the situation in Afghanistan against the fact that if there is a deterioration in the security situation , there will also be an increase in the number of Afghans in the refugee stream in Europe . After all , we have already had quite a lot of Afghan refugees in this refugee stream , and the Taliban 's activities in Kunduz , its takeover and so forth , are of course worrying .""}, … {2021,""immig"",0.055,""senti:negneg"",""Mr President ! It is undeniable that our main humanitarian groups — Somalis , Iraqis and Afghans — have been very poorly integrated . The basic Finnish solution is to do everything possible to combat this kind of immigration .""}]",50
"""MISC""","""inAfghanistan""","[{2018,""defen"",3.277,""senti:neupos"",""But let me remind you of what the Committee on Defence has said four years ago in its own opinion . It says : ‘ The committee considers it important that , after the end of the military prese

### Keywords and entities

In [14]:
# Look for a particular keyword associated to a particular entity
def search_keyword_for_entity(keyword, entity):
    pattern_entity = rf'(?i){entity}'
    pattern_keyword = rf'(?i){keyword}'

    return get_results(
        df.filter(
            pl.col('entity_content').str.contains(pattern_entity),
            pl.col('sentence_content_current').str.contains(pattern_keyword),
        )
    )


In [15]:
# Look for mentions of "war" keyword and "afghan" entity
search_keyword_for_entity(r'\bwar\b', 'afghan')

entity_category,entity_content,sentences_with_context,nb_sentences
str,str,list[struct[5]],u32
"""LOC""","""Afghanistan""","[{2015,""inter"",0.625,""senti:mixneg"",""Syria and Iraq are key countries of origin for the refugee crisis affecting Europe . The continuing civil war in Afghanistan has increased the number of Afghans seeking to enter Europe , and the number of asylum seekers seeking to enter Europe from Africa has also remained high . A lasting solution to the refugee crisis will only be achieved by calming the situation in the crisis areas .""}, {2016,""immig"",0.823,""senti:mixneg"",""In the state of justice , all offenders are also treated the same way , regardless of their background . A lot of people have come to Finland from warring and war - torn countries : Iraq , Afghanistan , Somalia and Syria . These are countries to which expulsion is not easy , especially if these countries refuse to accept a person .""}, … {2021,""inter"",0.037,""senti:negneg"",""The lesson to be learned from this failure in Afghanistan is that we must avoid these things in the future , and it will happen in such a way that we do not get involved in things that do not belong to us . After all , this Afghanistan was a sort of US backyard war and a world police operation in which it and its allies failed miserably . The U.S. made major mistakes in the operation , particularly through the number of civilian casualties caused by its drone attacks , which caused the people to turn against the regime and allowed the Taliban to win easily .""}]",18
"""MISC""","""Afghans""","[{2015,""inter"",0.625,""senti:mixneg"",""Syria and Iraq are key countries of origin for the refugee crisis affecting Europe . The continuing civil war in Afghanistan has increased the number of Afghans seeking to enter Europe , and the number of asylum seekers seeking to enter Europe from Africa has also remained high . A lasting solution to the refugee crisis will only be achieved by calming the situation in the crisis areas .""}, {2020,""lawcr"",0.292,""senti:negneg"",""Here 's a big report on the sexual crimes of foreigners , numbers and causes . This states that foreign citizens , especially Iraqis and Afghans , are overrepresented in sexual crime statistics , and experts say that the causes of the crimes are war , a violent culture of honour and age structure . There is one here , for example , so , Mr Kiljunen , you should not go unknowingly , just Google a few words .""}]",2
"""MISC""","""Afghan""","[{2018,""inter"",1.696,""senti:neuneg"",""We need a strong peace process alongside military crisis management operations and we need civilian crisis management to strongly support the security situation in Afghanistan that is currently very weak . I would like to ask the ministers : how do you see this , how can we bring a genuinely stronger peace process alongside the crisis management operation , because it was Taliban or Afghan , and war will not solve it ? Peace and peace , a political solution , is the only way .""}]",1


In [16]:
# Look for mentions of "war" keyword and "ukrain" entity
search_keyword_for_entity(r'\bwar\b', 'ukrain')

entity_category,entity_content,sentences_with_context,nb_sentences
str,str,list[struct[5]],u32
"""LOC""","""Ukraine""","[{2015,""inter"",4.24,""senti:mixpos"",""It is also more serious in our own security environment , as the new government 's government programme rightly states . However , as an organisation , the OSCE is the only one who in practice is able to influence the positive development of conditions in Ukraine , both in the face of its civilian operation and in the face of a war on the east side of Ukraine . This OSCE 's own reform work is being carried out because , however , the OSCE 's efforts are not sufficient when Europe is threatened by such serious crises .""}, {2015,""inter"",4.24,""senti:mixpos"",""It is also more serious in our own security environment , as the new government 's government programme rightly states . However , as an organisation , the OSCE is the only one who in practice is able to influence the positive development of conditions in Ukraine , both in the face of its civilian operation and in the face of a war on the east side of Ukraine . This OSCE 's own reform work is being carried out because , however , the OSCE 's efforts are not sufficient when Europe is threatened by such serious crises .""}, … {2021,""inter"",0.093,""senti:negneg"",""Russia is threatening , in practice , to start a war in Europe , a war against Ukraine . Of course , talking about the start of the war is somewhat misleading , as Russia attacked Ukraine as early as 2014 , when it illegally occupied the Crimean Peninsula and joined it and began an ongoing war in Eastern Ukraine . There is grotesque propaganda that Russia is currently presenting to its own people and to the whole world .""}]",27
"""LOC""","""Eastern Ukraine""","[{2015,""inter"",0.169,""senti:negneg"",""Whatever mistakes Ukraine has made in strengthening its own society , it is quite clear that Russia 's action in Ukraine is unacceptable , Robbery of Crimea and war in Eastern Ukraine . It is great that the Nordic Council has condemned Russia 's actions .""}, {2015,""inter"",0.131,""senti:negneg"",""However , it must be clearly stated that Russia 's action has caused the crisis in Ukraine . Russia has violated its agreements , including the Etyk Treaty , which was signed by its predecessor Soviet Union , Russia has attacked its neighbouring Ukraine , took over the Crimean Peninsula and is currently at war in Eastern Ukraine . At the same time , Russian propaganda lies to its own and to the rest of the world and even stirs up a kind of war psychology in Russia , where in propaganda NATO and the West are any day attacking Russia under the leadership of various fascists and Nazis .""}, … {2021,""inter"",0.093,""senti:negneg"",""Russia is threatening , in practice , to start a war in Europe , a war against Ukraine . Of course , talking about the start of the war is somewhat misleading , as Russia attacked Ukraine as early as 2014 , when it illegally occupied the Crimean Peninsula and joined it and began an ongoing war in Eastern Ukraine . There is grotesque propaganda that Russia is currently presenting to its own people and to the whole world .""}]",21
"""MISC""","""Ukrainian""","[{2016,""inter"",0.884,""senti:mixneg"",""A time of international political sensitivity would not need more excessive war rhetoric now than the display of military muscles . The Ukrainian war and the Syrian war have , in a typical way , also brought evidence desires elsewhere , including in the Baltic Sea region . Practice must be done , but there is always the risk that military technical exercises starting in Finland , for example , will be interpreted in public opinion beyond their purpose .""}, {2016,""defen"",3.727,""senti:mixpos"",""It has not been long since Finland ’s defence policy based on general military duty and defence of the country as a whole was still a source of wonder in many European countries . The invasion of Crimea and the beginning of the Ukrainian war at th